## Situation 4 — Chase Recovery (PPI Component 4)

### What this measures
How a batter performs when team loses early wickets in a chase
AND the team still wins the match.

This is the rarest and most valuable skill in ODI batting:
"Can you rescue a chase that looked lost?"

### Filter conditions
| Condition | Value | Reason |
|---|---|---|
| innings | == 2 | Chasing innings only |
| match_id | in won_chases | Only matches chase was won |
| cumulative_wickets | >= 3 | Early collapse happened |
| over_num | < 25 | Collapse was early in chase |
| wides | == 0 | Legal deliveries only |

### Why every ball here is already a winning contribution
We pre-filter to won_chases before anything else.
So every single ball in pressure_s4 = a ball faced during
a collapse in a chase that was eventually won.
Win% as a metric would be 100% for everyone — useless.
Instead we measure HOW MUCH they contributed during that collapse.

### Metric: s4_score = Average × Strike Rate / 100
| Component | Reason |
|---|---|
| s4_avg | Did they score enough runs to matter? |
| s4_sr | Did they score fast enough to keep chase alive? |
| combined | Both needed — slow 60 or fast 10 both insufficient |

### Why both average AND strike rate here
Unlike S1 (first innings, SR only) — in a chase recovery
you need BOTH:
- Enough runs (average) to actually win the match
- Fast enough (SR) because target clock is ticking

A batter scoring 80 off 120 balls in a chase recovery
is less valuable than 60 off 50 balls.

### Minimum qualification
5+ recovery innings — ensures meaningful sample size.

### Key difference from other situations
| Situation | Context | Metric |
|---|---|---|
| S1 | First innings collapse | SR only |
| S2 | Death overs chase | SR × Win% |
| S3 | World Cup | Avg + Consistency |
| S4 | Chase collapse recovery | Avg × SR |

S4 is unique because it combines both patience
AND aggression in the same innings.

In [1]:
import pandas as pd
import numpy as np
import os

# load master data - takes 3 seconds now
master_df = pd.read_parquet(r"D:\CricMetric-AI\data\processed\master_odi.parquet")

print(f"Shape: {master_df.shape}")
print(f"Columns: {master_df.columns.tolist()}")

Shape: (1349408, 32)
Columns: ['innings', 'over', 'batting_team', 'batter', 'non_striker', 'bowler', 'runs_off_bat', 'extras', 'wides', 'noballs', 'byes', 'legbyes', 'penalty', 'wicket_type', 'player_dismissed', 'other_wicket_type', 'other_player_dismissed', 'running_total', 'extra1', 'extra2', 'extra3', 'match_id', 'match_date', 'team1', 'team2', 'venue', 'event', 'toss_winner', 'city', 'season', 'over_num', 'ball_num']


In [2]:
cols_to_drop = [
    'extra1', 'extra2', 'extra3',
    'running_total', 'penalty',
    'other_wicket_type', 'other_player_dismissed'
]

master_df = master_df.drop(columns=cols_to_drop)

print(f"Remaining columns: {master_df.shape[1]}")
print(master_df.columns.tolist())

Remaining columns: 25
['innings', 'over', 'batting_team', 'batter', 'non_striker', 'bowler', 'runs_off_bat', 'extras', 'wides', 'noballs', 'byes', 'legbyes', 'wicket_type', 'player_dismissed', 'match_id', 'match_date', 'team1', 'team2', 'venue', 'event', 'toss_winner', 'city', 'season', 'over_num', 'ball_num']


In [3]:
# check null values in each column
null_counts = master_df.isnull().sum()
null_percent = (null_counts / len(master_df) * 100).round(2)

null_df = pd.DataFrame({
    'null_count': null_counts,
    'null_percent': null_percent
})

# only show columns that have nulls
null_df = null_df[null_df['null_count'] > 0]
print(null_df)

Empty DataFrame
Columns: [null_count, null_percent]
Index: []


In [4]:
# check for duplicate balls (same match, same over, same ball number)
duplicates = master_df.duplicated(
    subset=['match_id', 'over_num', 'ball_num', 'innings']
).sum()

print(f"Duplicate balls: {duplicates}")

Duplicate balls: 0


In [5]:

# every match → every innings → every over → every ball in order
master_df = master_df.sort_values(
    ['match_id', 'innings', 'over_num', 'ball_num']
).reset_index(drop=True)

print("Sorted successfully")
print(master_df[['match_id', 'innings', 'over_num', 'ball_num', 'batter', 'runs_off_bat']].head(10))

Sorted successfully
  match_id innings  over_num  ball_num     batter  runs_off_bat
0  1000887       1         0         1  DA Warner             0
1  1000887       1         0         2  DA Warner             0
2  1000887       1         0         3  DA Warner             0
3  1000887       1         0         4  DA Warner             0
4  1000887       1         0         5  DA Warner             0
5  1000887       1         0         6  DA Warner             0
6  1000887       1         0         7  DA Warner             0
7  1000887       1         1         1    TM Head             0
8  1000887       1         1         2    TM Head             1
9  1000887       1         1         3  DA Warner             0


In [7]:
print(master_df['wicket_type'].value_counts().head(20))
print("\nUnique values sample:")
print(master_df['wicket_type'].unique()[:20])

wicket_type
""                       1312494
caught                     21118
bowled                      6915
lbw                         4102
run out                     2690
caught and bowled           1108
stumped                      882
retired hurt                  57
hit wicket                    35
obstructing the field          6
"caught                        1
Name: count, dtype: int64

Unique values sample:
<ArrowStringArray>
[                   '""',                'bowled',                'caught',
               'run out',                   'lbw',          'retired hurt',
               'stumped',     'caught and bowled',            'hit wicket',
 'obstructing the field',               '"caught']
Length: 11, dtype: str


In [8]:
# flag each ball as wicket or not
# recompute is_wicket correctly after sorting
master_df['is_wicket'] = master_df['wicket_type'].apply(
    lambda x: 1 if x not in ['', '""'] else 0
)

# verify - should show mostly 0s with occasional 1s
print("Wicket distribution:")
print(master_df['is_wicket'].value_counts())

# recompute cumulative wickets
master_df['cumulative_wickets'] = master_df.groupby(
    ['match_id', 'innings']
)['is_wicket'].cumsum()

# check first match again
print("\nFirst 15 balls:")
print(master_df[['match_id', 'innings', 'over_num', 'ball_num',
                  'batter', 'runs_off_bat', 'is_wicket',
                  'cumulative_runs', 'cumulative_wickets']].head(15))

# compute total runs per ball (batting runs + extras)
master_df['total_runs_ball'] = master_df['runs_off_bat'] + master_df['extras']

# cumulative runs and wickets within each innings of each match
master_df['cumulative_runs'] = master_df.groupby(
    ['match_id', 'innings']
)['total_runs_ball'].cumsum()

master_df['cumulative_wickets'] = master_df.groupby(
    ['match_id', 'innings']
)['is_wicket'].cumsum()

# verify
print(master_df[['match_id', 'innings', 'over_num', 'ball_num', 
                  'batter', 'runs_off_bat', 'is_wicket',
                  'cumulative_runs', 'cumulative_wickets']].head(15))

Wicket distribution:
is_wicket
0    1312494
1      36914
Name: count, dtype: int64

First 15 balls:
   match_id innings  over_num  ball_num     batter  runs_off_bat  is_wicket  \
0   1000887       1         0         1  DA Warner             0          0   
1   1000887       1         0         2  DA Warner             0          0   
2   1000887       1         0         3  DA Warner             0          0   
3   1000887       1         0         4  DA Warner             0          0   
4   1000887       1         0         5  DA Warner             0          0   
5   1000887       1         0         6  DA Warner             0          0   
6   1000887       1         0         7  DA Warner             0          0   
7   1000887       1         1         1    TM Head             0          0   
8   1000887       1         1         2    TM Head             1          0   
9   1000887       1         1         3  DA Warner             0          0   
10  1000887       1         1  

In [9]:
print("Wicket distribution:")
print(master_df['is_wicket'].value_counts())

Wicket distribution:
is_wicket
0    1312494
1      36914
Name: count, dtype: int64


In [10]:
master_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1349408 entries, 0 to 1349407
Data columns (total 29 columns):
 #   Column              Non-Null Count    Dtype
---  ------              --------------    -----
 0   innings             1349408 non-null  str  
 1   over                1349408 non-null  str  
 2   batting_team        1349408 non-null  str  
 3   batter              1349408 non-null  str  
 4   non_striker         1349408 non-null  str  
 5   bowler              1349408 non-null  str  
 6   runs_off_bat        1349408 non-null  int64
 7   extras              1349408 non-null  int64
 8   wides               1349408 non-null  int64
 9   noballs             1349408 non-null  int64
 10  byes                1349408 non-null  str  
 11  legbyes             1349408 non-null  str  
 12  wicket_type         1349408 non-null  str  
 13  player_dismissed    1349408 non-null  str  
 14  match_id            1349408 non-null  str  
 15  match_date          1349408 non-null  str  
 16  team1      

In [14]:
# string to int conversions
int_cols = ['innings', 'over_num', 'ball_num', 'byes', 'legbyes']

for col in int_cols:
    master_df[col] = pd.to_numeric(master_df[col], errors='coerce').fillna(0).astype(int)

# match_date to datetime
master_df['match_date'] = pd.to_datetime(master_df['match_date'], errors='coerce')

# extract year as separate column
master_df['year'] = master_df['match_date'].dt.year

master_df.info()



<class 'pandas.DataFrame'>
RangeIndex: 1349408 entries, 0 to 1349407
Data columns (total 30 columns):
 #   Column              Non-Null Count    Dtype         
---  ------              --------------    -----         
 0   innings             1349408 non-null  int64         
 1   over                1349408 non-null  str           
 2   batting_team        1349408 non-null  str           
 3   batter              1349408 non-null  str           
 4   non_striker         1349408 non-null  str           
 5   bowler              1349408 non-null  str           
 6   runs_off_bat        1349408 non-null  int64         
 7   extras              1349408 non-null  int64         
 8   wides               1349408 non-null  int64         
 9   noballs             1349408 non-null  int64         
 10  byes                1349408 non-null  int64         
 11  legbyes             1349408 non-null  int64         
 12  wicket_type         1349408 non-null  str           
 13  player_dismissed    134

In [15]:
master_df.to_parquet(
    r"D:\CricMetric-AI\data\processed\master_odi.parquet",
    index=False
)
print("Saved cleaned dataframe successfully")
print(f"Shape: {master_df.shape}")

Saved cleaned dataframe successfully
Shape: (1349408, 30)


## Situation 1 — Early Collapse (Pressure Performance Index Component 1)

### What this measures
How a batter performs when the team is in trouble early in the innings.
Specifically: balls faced when 3+ wickets have already fallen in first 20 overs.

### Why this matters
Any batter can score when openers put on 150.
The truly great batters score when the team is 3 down for 40.
This separates match-savers from fair-weather run-scorers.

### Filter conditions
| Condition | Value | Reason |
|---|---|---|
| cumulative_wickets | >= 3 | Team in collapse |
| over_num | < 20 | Early innings only |
| innings | == 1 | First innings pressure |
| wides | == 0 | Legal deliveries only |

### Metric used
Strike Rate under pressure (runs per 100 balls)
In early collapse, scoring speed matters — 
a batter who scores at 80 SR stabilizes AND accelerates the innings.

### Minimum qualification
50+ balls faced in pressure situations.
Removes batters with small sample sizes.

### Key finding
286 batters qualified with 50+ pressure balls.
Legends like Dhoni (531 balls), Sangakkara (661 balls) 
faced the most pressure — reflecting their middle order roles.
Kohli (164 balls) faced less — reflecting his top order position
where team rarely collapsed before he arrived.

In [16]:
# Situation 1: batting when 3+ wickets down in first 20 overs
pressure_s1 = master_df[
    (master_df['cumulative_wickets'] >= 3) &
    (master_df['over_num'] < 20) &
    (master_df['innings'] == 1) &
    (master_df['wides'] == 0)  # only legal deliveries
].copy()

print(f"Pressure situation 1 balls: {len(pressure_s1)}")
print(f"Unique batters in pressure: {pressure_s1['batter'].nunique()}")

# runs and balls per batter in situation 1
s1_stats = pressure_s1.groupby('batter').agg(
    s1_runs=('runs_off_bat', 'sum'),
    s1_balls=('runs_off_bat', 'count')
).reset_index()

s1_stats['s1_sr'] = (s1_stats['s1_runs'] / s1_stats['s1_balls'] * 100).round(2)

# only keep batters with minimum 50 balls in pressure
s1_stats = s1_stats[s1_stats['s1_balls'] >= 50]

print(f"\nBatters with 50+ pressure balls (S1): {len(s1_stats)}")
print("\nTop 10 by strike rate under pressure:")
print(s1_stats.sort_values('s1_sr', ascending=False).head(10))

Pressure situation 1 balls: 53595
Unique batters in pressure: 880

Batters with 50+ pressure balls (S1): 286

Top 10 by strike rate under pressure:
          batter  s1_runs  s1_balls   s1_sr
162  CJ Anderson      159       108  147.22
364     JD Ryder      100        78  128.21
386   JM Davison       71        60  118.33
588  NLTC Perera      107        93  115.05
501  MDKJ Perera      105        98  107.14
839     V Sehwag       53        50  106.00
201    DA Warner      118       115  102.61
121  BJ McMullen       77        78   98.72
165  CJ Ferguson       72        73   98.63
518    ML Hayden       89        91   97.80


In [17]:
# check specific players
check_players = ['V Kohli', 'RG Sharma', 'MS Dhoni', 'AB de Villiers', 'KC Sangakkara']

for player in check_players:
    player_data = pressure_s1[pressure_s1['batter'] == player]
    print(f"{player}: {len(player_data)} pressure balls")

V Kohli: 164 pressure balls
RG Sharma: 202 pressure balls
MS Dhoni: 531 pressure balls
AB de Villiers: 323 pressure balls
KC Sangakkara: 661 pressure balls


In [21]:
# Situation 2: overs 40-50, chasing 270+, innings 2
# first find matches where team 2 needed 270+
innings2 = master_df[master_df['innings'] == 2].copy()

# get target for each match (max cumulative runs in innings 1 + 1)
innings1_totals = master_df[
    master_df['innings'] == 1
].groupby('match_id')['cumulative_runs'].max().reset_index()
innings1_totals.columns = ['match_id', 'innings1_total']
innings1_totals['target'] = innings1_totals['innings1_total'] + 1

# merge target into innings 2
innings2 = innings2.merge(innings1_totals, on='match_id', how='left')

# filter: overs 40+, target 270+, legal deliveries only
pressure_s2 = innings2[
    (innings2['over_num'] >= 40) &
    (innings2['target'] >= 270) &
    (innings2['wides'] == 0)
].copy()

print(f"Pressure situation 2 balls: {len(pressure_s2)}")
print(f"Unique batters: {pressure_s2['batter'].nunique()}")

# compute stats
s2_stats = pressure_s2.groupby('batter').agg(
    s2_runs=('runs_off_bat', 'sum'),
    s2_balls=('runs_off_bat', 'count')
).reset_index()

s2_stats['s2_sr'] = (s2_stats['s2_runs'] / s2_stats['s2_balls'] * 100).round(2)
s2_stats = s2_stats[s2_stats['s2_balls'] >= 50]

print(f"\nBatters with 30+ death chase balls: {len(s2_stats)}")
print("\nTop 10 by death chase strike rate:")
print(s2_stats.sort_values('s2_sr', ascending=False).head(10))

Pressure situation 2 balls: 33356
Unique batters: 931

Batters with 30+ death chase balls: 204

Top 10 by death chase strike rate:
             batter  s2_runs  s2_balls   s2_sr
511    MG Bracewell      151        81  186.42
496        MA Leask      110        61  180.33
814   Shahid Afridi      209       117  178.63
535      MP Stoinis      116        65  178.46
253    Fakhar Zaman      123        69  178.26
744       SB Styris      108        63  171.43
193   DAS Gunaratne      122        72  169.44
431    KP Pietersen      132        79  167.09
332  Iftikhar Ahmed       88        53  166.04
670      R Shepherd      108        66  163.64


In [22]:
print(innings2.shape)
print(innings2.columns.tolist())

(616034, 32)
['innings', 'over', 'batting_team', 'batter', 'non_striker', 'bowler', 'runs_off_bat', 'extras', 'wides', 'noballs', 'byes', 'legbyes', 'wicket_type', 'player_dismissed', 'match_id', 'match_date', 'team1', 'team2', 'venue', 'event', 'toss_winner', 'city', 'season', 'over_num', 'ball_num', 'is_wicket', 'total_runs_ball', 'cumulative_runs', 'cumulative_wickets', 'year', 'innings1_total', 'target']


In [ ]:
# get match winner for each innings 2
match_outcomes = innings2.groupby('match_id').agg(
    chasing_team=('batting_team', 'first'),
    final_runs=('cumulative_runs', 'max'),
    target=('target', 'first')
).reset_index()

# did chasing team win?
match_outcomes['chase_won'] = (
    match_outcomes['final_runs'] >= match_outcomes['target']
).astype(int)

print(f"Total chase matches: {len(match_outcomes)}")
print(f"Chases won: {match_outcomes['chase_won'].sum()}")
print(f"Chase win rate: {match_outcomes['chase_won'].mean()*100:.1f}%")

# updated filter - 250+ target
pressure_s2 = innings2[
    (innings2['over_num'] >= 40) &
    (innings2['target'] >= 250) &
    (innings2['wides'] == 0)
].copy()

print(f"\nPressure situation 2 balls: {len(pressure_s2)}")
print(f"Unique batters: {pressure_s2['batter'].nunique()}")

# merge win result into pressure_s2
pressure_s2 = pressure_s2.merge(
    match_outcomes[['match_id', 'chase_won']],
    on='match_id',
    how='left'
)

# per batter per match death chase stats
s2_innings = pressure_s2.groupby(
    ['batter', 'match_id']
).agg(
    death_runs=('runs_off_bat', 'sum'),
    death_balls=('runs_off_bat', 'count'),
    chase_won=('chase_won', 'first')
).reset_index()

# per batter overall death chase stats
s2_stats = s2_innings.groupby('batter').agg(
    s2_innings=('match_id', 'count'),
    s2_runs=('death_runs', 'sum'),
    s2_balls=('death_balls', 'sum'),
    s2_wins=('chase_won', 'sum')
).reset_index()

# compute metrics
s2_stats['s2_sr'] = (
    s2_stats['s2_runs'] / s2_stats['s2_balls'] * 100
).round(2)

s2_stats['s2_win_pct'] = (
    s2_stats['s2_wins'] / s2_stats['s2_innings'] * 100
).round(2)

# combined score
s2_stats['s2_score'] = (
    s2_stats['s2_sr'] * s2_stats['s2_win_pct'] / 100
).round(2)

# updated minimum - 5 innings
s2_stats = s2_stats[s2_stats['s2_innings'] >= 8]

print(f"\nBatters with 8+ death chase innings: {len(s2_stats)}")
print("\nTop 10 by combined death chase score:")
print(s2_stats.sort_values('s2_score', ascending=False)[
    ['batter', 's2_innings', 's2_runs', 's2_sr', 's2_win_pct', 's2_score']
].head(10))

Total chase matches: 2477
Chases won: 1180
Chase win rate: 47.6%

Pressure situation 2 balls: 42108
Unique batters: 1016

Batters with 7+ death chase innings: 133

Top 10 by combined death chase score:
             batter  s2_innings  s2_runs   s2_sr  s2_win_pct  s2_score
966         V Kohli          24      516  146.18       83.33    121.81
469        KL Rahul           8      149  134.23       87.50    117.45
261      EJG Morgan          14      293  141.55       78.57    111.22
829        SK Raina          22      417  131.55       81.82    107.63
1005   Yuvraj Singh          20      397  122.53       80.00     98.02
460   KC Sangakkara          12      230  128.49       75.00     96.37
395         JE Root          12      232  128.18       75.00     96.14
746       RG Sharma          12      241  141.76       66.67     94.51
945      TWM Latham          10      189  117.39       80.00     93.91
849       SPD Smith          11      196  113.95       81.82     93.23


## Situation 2 — Death Chase (Revised Metric)

### Why we changed from pure Strike Rate to Win Contribution Score

Initial approach used Strike Rate in overs 40-50 when chasing 270+ as the pressure metric.

**Problem identified:** Strike Rate alone is misleading in chase situations.
- A batter scoring 80 off 40 balls in a comfortable chase (need 100 off 60) 
  is NOT the same as scoring 80 off 40 balls when team needs 120 off 40.
- High SR in a losing chase means nothing.
- What actually matters = did the team WIN the match when this batter 
  performed in death overs?

### New metric: Death Chase Score = Strike Rate × Win %

| Component | What it measures |
|---|---|
| `s2_sr` | How fast did they score in death overs? |
| `s2_win_pct` | How often did team win those matches? |
| `s2_score` | Combined — fast scoring that led to wins |

**Example:**
- Batter A: SR 180, but team lost 70% of those matches → s2_score = 54
- Batter B: SR 140, team won 80% of those matches → s2_score = 112
- Batter B is the better death chase performer despite lower SR

### Cricket insight
A true death chase specialist wins matches, not just scores fast.
This metric rewards match-winning contributions over empty statistics.

In [31]:
import glob as gl

WC_PATH = r"D:\CricMetric-AI\data\raw\worldcup"
wc_files = gl.glob(os.path.join(WC_PATH, "*.csv"))

wc_match_ids = [
    os.path.basename(f).replace('.csv', '')
    for f in wc_files
]

print(f"World Cup match IDs loaded: {len(wc_match_ids)}")

World Cup match IDs loaded: 265


## Situation 3 — World Cup / Tournament Performance (PPI Component 3)

### What this measures
How a batter performs in ICC World Cup matches specifically.
World Cup = highest pressure tournament in ODI cricket.
Every match matters more — knockout stages, nation expectations, legacy moments.

### Why NOT strike rate here
Unlike death chases, World Cup innings have varying contexts:
- A team chasing 180 doesn't need 150 SR
- A team defending needs a solid 60-ball 50
- What matters = did they score runs consistently AND win matches?

### Metrics used
| Metric | Description |
|---|---|
| s3_avg | Average runs per dismissal in WC matches |
| s3_consistency | % of innings where they scored 30+ |
| s3_50plus | Number of 50+ scores in WC matches |

### Minimum qualification
10+ World Cup innings (ensures meaningful sample)

### Expected finding
Players like Kohli, Rohit, Sangakkara should dominate
given their World Cup records.

In [34]:
# Situation 3: World Cup performance
pressure_s3 = master_df[
    (master_df['match_id'].isin(wc_match_ids)) &
    (master_df['wides'] == 0)
].copy()

print(f"World Cup balls: {len(pressure_s3)}")
print(f"Unique batters: {pressure_s3['batter'].nunique()}")

# runs per innings in WC
s3_innings = pressure_s3.groupby(
    ['batter', 'match_id', 'innings']
).agg(
    innings_runs=('runs_off_bat', 'sum'),
    innings_balls=('runs_off_bat', 'count'),
    got_out=('is_wicket', 'max')
).reset_index()

# per batter WC stats
s3_stats = s3_innings.groupby('batter').agg(
    s3_innings=('match_id', 'count'),
    s3_runs=('innings_runs', 'sum'),
    s3_dismissals=('got_out', 'sum'),
    s3_50plus=('innings_runs', lambda x: (x >= 50).sum())
).reset_index()

# average
s3_stats['s3_avg'] = (
    s3_stats['s3_runs'] /
    s3_stats['s3_dismissals'].replace(0, 1)
).round(2)

# consistency
s3_consistency = s3_innings.groupby('batter').apply(
    lambda x: (x['innings_runs'] >= 30).sum() / len(x) * 100
).reset_index()
s3_consistency.columns = ['batter', 's3_consistency']
s3_consistency['s3_consistency'] = s3_consistency['s3_consistency'].round(2)

s3_stats = s3_stats.merge(s3_consistency, on='batter')

# minimum 5 WC innings
s3_stats = s3_stats[s3_stats['s3_innings'] >= 10]

print(f"\nBatters with 10+ WC innings: {len(s3_stats)}")
print("\nTop 10 by World Cup average:")
print(s3_stats.sort_values('s3_avg', ascending=False)[
    ['batter', 's3_innings', 's3_runs', 's3_avg', 
     's3_consistency', 's3_50plus']
].head(10))

World Cup balls: 137817
Unique batters: 695

Batters with 10+ WC innings: 145

Top 10 by World Cup average:
             batter  s3_innings  s3_runs  s3_avg  s3_consistency  s3_50plus
11        A Symonds          13      515   85.83           38.46          4
230        HH Gibbs          14      726   72.60           71.43          7
17   AB de Villiers          22     1207   71.00           59.09         10
429     Mahmudullah          20      894   68.77           50.00          6
403       MJ Clarke          21      888   63.43           57.14          8
592         SS Iyer          10      505   63.12           60.00          5
523       RG Sharma          26     1443   62.74           69.23         12
342   KS Williamson          24     1055   62.06           66.67          7
541      RT Ponting          25     1160   61.05           52.00          9
337        KL Rahul          18      783   60.23           55.56          6


In [37]:
kohli_wc = s3_stats[s3_stats['batter'] == 'V Kohli']
dhoni_wc=s3_stats[s3_stats['batter'] == 'MS Dhoni']
print(kohli_wc)
print(dhoni_wc)

      batter  s3_innings  s3_runs  s3_dismissals  s3_50plus  s3_avg  \
664  V Kohli          35     1673             28         15   59.75   

     s3_consistency  
664           65.71  
       batter  s3_innings  s3_runs  s3_dismissals  s3_50plus  s3_avg  \
424  MS Dhoni          24      752             17          5   44.24   

     s3_consistency  
424           45.83  


## Situation 4 — Chase Recovery (PPI Component 4)

### What this measures
How a batter performs when team loses early wickets in a chase
BUT still wins the match.
This captures the "match-saving chaser" — the most valuable
batting skill in ODI cricket.

### Filter conditions
| Condition | Value | Reason |
|---|---|---|
| innings | == 2 | Chasing innings only |
| cumulative_wickets | >= 3 | Early collapse |
| over_num | < 25 | Early in chase |
| wides | == 0 | Legal deliveries only |
| match result | chase won | Only winning recoveries |

### Metric used
Average runs per innings in recovery situations.
Strike rate matters less here — survival + scoring = winning.

### Why this is different from S1
S1 = first innings collapse, rebuild the total
S4 = second innings collapse, win the match despite collapse
S4 is harder — you have a target, a clock, and no margin for error.

In [38]:
# first get matches where chase was won
# we already have match_outcomes from S2
won_chases = match_outcomes[
    match_outcomes['chase_won'] == 1
]['match_id'].tolist()

print(f"Total won chases: {len(won_chases)}")

# filter: innings 2, early collapse, won matches only
pressure_s4 = innings2[
    (innings2['cumulative_wickets'] >= 3) &
    (innings2['over_num'] < 25) &
    (innings2['match_id'].isin(won_chases)) &
    (innings2['wides'] == 0)
].copy()

print(f"S4 pressure balls: {len(pressure_s4)}")
print(f"Unique batters: {pressure_s4['batter'].nunique()}")

# runs per innings per batter in S4
s4_innings = pressure_s4.groupby(
    ['batter', 'match_id']
).agg(
    s4_runs=('runs_off_bat', 'sum'),
    s4_balls=('runs_off_bat', 'count'),
    got_out=('is_wicket', 'max')
).reset_index()

# per batter overall S4 stats
s4_stats = s4_innings.groupby('batter').agg(
    s4_innings=('match_id', 'count'),
    s4_total_runs=('s4_runs', 'sum'),
    s4_total_balls=('s4_balls', 'sum'),
    s4_dismissals=('got_out', 'sum')
).reset_index()

# average in recovery situations
s4_stats['s4_avg'] = (
    s4_stats['s4_total_runs'] /
    s4_stats['s4_dismissals'].replace(0, 1)
).round(2)

# strike rate too
s4_stats['s4_sr'] = (
    s4_stats['s4_total_runs'] /
    s4_stats['s4_total_balls'] * 100
).round(2)

# combined score avg × SR / 100
s4_stats['s4_score'] = (
    s4_stats['s4_avg'] * s4_stats['s4_sr'] / 100
).round(2)

# minimum 5 recovery innings
s4_stats = s4_stats[s4_stats['s4_innings'] >= 5]

print(f"\nBatters with 5+ recovery innings: {len(s4_stats)}")
print("\nTop 10 by chase recovery score:")
print(s4_stats.sort_values('s4_score', ascending=False)[
    ['batter', 's4_innings', 's4_total_runs',
     's4_avg', 's4_sr', 's4_score']
].head(10))

Total won chases: 1180
S4 pressure balls: 29473
Unique batters: 554

Batters with 5+ recovery innings: 123

Top 10 by chase recovery score:
              batter  s4_innings  s4_total_runs  s4_avg   s4_sr  s4_score
3         A Flintoff          11            255   85.00  118.60    100.81
265     KIC Asalanka           8            218  109.00   86.51     94.30
12    AB de Villiers          18            376   94.00   96.91     91.10
260       KA Pollard           7            150   75.00  104.17     78.13
320     MG Bracewell           5             85   85.00   90.43     76.87
240  JN Loftie-Eaton           6             87   87.00   87.88     76.46
268         KL Rahul           6            116  116.00   61.38     71.20
69       BB McCullum           5             98   98.00   68.53     67.16
349      Mahmudullah          10            211   70.33   91.74     64.52
178        H Klaasen           6            143   47.67  134.91     64.31


## Final PPI — Combining All 4 Pressure Situations

### Weighting logic
| Situation | Weight | Reason |
|---|---|---|
| S1 Early collapse SR | 25% | Common situation, large sample |
| S2 Death chase score | 30% | Match-winning impact, highest value |
| S3 World Cup avg | 25% | Tournament pressure, legacy defining |
| S4 Chase recovery | 20% | Rarest skill, smaller sample |

### Normalisation
Each component normalised to 0-100 scale using MinMaxScaler.
Prevents one metric dominating due to different value ranges.
Example: s2_score ranges 0-137, s3_avg ranges 0-85
Without normalisation, S2 would dominate unfairly.

### Missing data handling
Batters missing from a situation → filled with population mean.
This gives them a neutral score, not zero.
Zero would unfairly punish batters who never played
in that specific situation.

### Minimum qualification
500+ total legal balls faced in career.
Removes batters with tiny sample sizes.

In [42]:
from sklearn.preprocessing import MinMaxScaler

# get all batters with 500+ career balls
total_balls = master_df[
    master_df['wides'] == 0
].groupby('batter')['runs_off_bat'].count().reset_index()
total_balls.columns = ['batter', 'total_balls']
qualified = total_balls[total_balls['total_balls'] >= 500]

print(f"Batters with 500+ career balls: {len(qualified)}")

# start PPI dataframe with qualified batters
ppi_df = qualified.copy()

# merge all 4 situations
ppi_df = ppi_df.merge(
    s1_stats[['batter', 's1_sr']], 
    on='batter', how='left'
)
ppi_df = ppi_df.merge(
    s2_stats[['batter', 's2_score']], 
    on='batter', how='left'
)
ppi_df = ppi_df.merge(
    s3_stats[['batter', 's3_avg', 's3_consistency']], 
    on='batter', how='left'
)
ppi_df = ppi_df.merge(
    s4_stats[['batter', 's4_score']], 
    on='batter', how='left'
)

print(f"\nBefore filling nulls:")
print(ppi_df.isnull().sum())

# fill missing with population mean
ppi_df['s1_sr'] = ppi_df['s1_sr'].fillna(ppi_df['s1_sr'].mean())
ppi_df['s2_score'] = ppi_df['s2_score'].fillna(ppi_df['s2_score'].mean())
ppi_df['s3_avg'] = ppi_df['s3_avg'].fillna(ppi_df['s3_avg'].mean())
ppi_df['s3_consistency'] = ppi_df['s3_consistency'].fillna(
    ppi_df['s3_consistency'].mean()
)
ppi_df['s4_score'] = ppi_df['s4_score'].fillna(ppi_df['s4_score'].mean())

print(f"\nAfter filling nulls:")
print(ppi_df.isnull().sum())




Batters with 500+ career balls: 526

Before filling nulls:
batter              0
total_balls         0
s1_sr             260
s2_score          432
s3_avg            387
s3_consistency    387
s4_score          403
dtype: int64

After filling nulls:
batter            0
total_balls       0
s1_sr             0
s2_score          0
s3_avg            0
s3_consistency    0
s4_score          0
dtype: int64


In [43]:
# normalise each component to 0-100
scaler = MinMaxScaler(feature_range=(0, 100))

ppi_df['s1_norm'] = scaler.fit_transform(ppi_df[['s1_sr']])
ppi_df['s2_norm'] = scaler.fit_transform(ppi_df[['s2_score']])
ppi_df['s3_norm'] = scaler.fit_transform(ppi_df[['s3_avg']])
ppi_df['s3_con_norm'] = scaler.fit_transform(ppi_df[['s3_consistency']])
ppi_df['s4_norm'] = scaler.fit_transform(ppi_df[['s4_score']])

# weighted PPI
ppi_df['PPI'] = (
    0.25 * ppi_df['s1_norm'] +
    0.30 * ppi_df['s2_norm'] +
    0.20 * ppi_df['s3_norm'] +
    0.05 * ppi_df['s3_con_norm'] +
    0.20 * ppi_df['s4_norm']
).round(2)

# sort by PPI
ppi_df = ppi_df.sort_values('PPI', ascending=False).reset_index(drop=True)
ppi_df['PPI_rank'] = ppi_df.index + 1

print(f"\nTotal qualified batters: {len(ppi_df)}")
print("\nTop 20 by PPI:")
print(ppi_df[['PPI_rank', 'batter', 'PPI', 
              's1_norm', 's2_norm', 
              's3_norm', 's4_norm',
              'total_balls']].head(20))


Total qualified batters: 526

Top 20 by PPI:
    PPI_rank           batter    PPI     s1_norm     s2_norm     s3_norm  \
0          1         KL Rahul  71.59   29.790542   96.420655   69.227071   
1          2          V Kohli  63.92   40.909835  100.000000   68.650078   
2          3   AB de Villiers  59.13   31.729668   56.908300   82.173338   
3          4          JE Root  53.53   43.937162   78.926197   50.342589   
4          5        RG Sharma  53.38   27.745050   77.588047   72.244260   
5          6         MS Dhoni  51.32   23.768614   57.228471   50.006010   
6          7     Yuvraj Singh  51.20   34.871543   80.469584   55.968265   
7          8      CJ Anderson  51.05  100.000000   36.011263   40.299185   
8          9        DA Miller  49.97   39.821633   50.841474   56.328886   
9         10    KC Sangakkara  49.46   31.034201   79.115015   67.327804   
10        11         SK Raina  49.21   30.682376   88.358920   40.299185   
11        12      BB McCullum  48.74   20.

In [44]:
print(ppi_df.columns.tolist())

['batter', 'total_balls', 's1_sr', 's2_score', 's3_avg', 's3_consistency', 's4_score', 's1_norm', 's2_norm', 's3_norm', 's3_con_norm', 's4_norm', 'PPI', 'PPI_rank']


In [45]:
kohli = ppi_df[ppi_df['batter'] == 'V Kohli']
print(kohli[['batter', 'PPI', 's1_norm', 's2_norm', 
             's3_norm', 's3_con_norm', 's4_norm']].to_string())

    batter    PPI    s1_norm  s2_norm    s3_norm  s3_con_norm    s4_norm
1  V Kohli  63.92  40.909835    100.0  68.650078    85.426417  28.462585


In [46]:
# show all columns properly
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

print(ppi_df[['PPI_rank', 'batter', 'PPI', 
              's1_norm', 's2_norm', 
              's3_norm', 's3_con_norm',
              's4_norm',
              'total_balls']].head(20).to_string())

    PPI_rank           batter    PPI     s1_norm     s2_norm     s3_norm  s3_con_norm     s4_norm  total_balls
0          1         KL Rahul  71.59   29.790542   96.420655   69.227071    72.230889   88.775510         3596
1          2          V Kohli  63.92   40.909835  100.000000   68.650078    85.426417   28.462585        15652
2          3   AB de Villiers  59.13   31.729668   56.908300   82.173338    76.820073   69.272109         9323
3          4          JE Root  53.53   43.937162   78.926197   50.342589    62.181487   28.469388         8415
4          5        RG Sharma  53.38   27.745050   77.588047   72.244260    90.002600   21.095238        12300
5          6         MS Dhoni  51.32   23.768614   57.228471   50.006010    59.581383   76.142857        11820
6          7     Yuvraj Singh  51.20   34.871543   80.469584   55.968265    68.096724   18.727891         7889
7          8      CJ Anderson  51.05  100.000000   36.011263   40.299185    48.993535   23.661578         1012
8

In [47]:
ppi_df.to_csv(
    r"D:\CricMetric-AI\data\processed\ppi_scores.csv",
    index=False
)
print(f"Saved. Total batters: {len(ppi_df)}")

Saved. Total batters: 526
